(p1-theory-end2end-ml-project)=
# P1 이론: 머신러닝 프로젝트

**감사의 글**

오렐리앙 제롱<font size='2'>Aurélien Géron</font>의 [Hands-On Machine Learning with Scikit-Learn and PyTorch (O'Reilly, 2025)](https://github.com/ageron/handson-mlp)에 사용된 코드를 참고한 강의노트이다. 보다 심화된 이해를 위해 책 원본을 읽을 것을 강력하게 권장한다. 자료를 공개한 오렐리앙 제롱과 일부 그림 자료를 제공해 준 한빛아카데미에게 진심어린 감사를 전한다.

**주요 내용**

캘리포니아 주택 가격 데이터를 이용하여 **머신러닝 프로젝트의 전체 흐름**을 살펴본다.

최종적으로 다음 질문에 답할 수 있게 된다.

> **실제 데이터를 이용해 머신러닝 문제를 어떻게 정의하고, 
> 어떤 데이터를 활용하고, 어떤 모델을 선택하고, 새로운 데이터에 대해 어떻게 평가할 것인가?**

프로젝트의 전체 흐름은 다음과 같다.

> **문제 정의 → 데이터 이해 → 훈련셋과 테스트셋 분리 → 탐색적 데이터분석 → 데이터 준비 → 모델 훈련 → 모델 비교 → 최종 평가**

## 머신러닝 프로젝트와 데이터 분석

### 문제 정의와 데이터

머신러닝 프로젝트는 모델을 고르는 것보다 **무엇을 예측하려는지 명확히 하는 것**에서 시작한다.

여기서는 1990년 미국 캘리포니아의 20,640개 구역에 대한 데이터를 사용한다.
각 구역의 데이터는 
경도, 위도, 주택 건물 중위연령, 총 방 수, 총 침실 수, 인구, 가구 수, 중위소득, 중위 주택가격, 해안 근접도
등 총 10개의 특성이 포함되어 있다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/housing-data.png?raw=true" width="700">
</div>

캘리포니아는 미국 서부에 위치하며, 1990년 당시 2,976만명의 인구를 가졌다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/LA-USA01.png?raw=true" width="600">
</div>

이 프로젝트의 목표는 다른 특성을 이용하여 각 구역의 **중위 주택가격**을 예측하는 것이다.

- **특성**(feature): 중위 주택가격을 제외한 입력 정보
- **타깃**(target): 중위 주택가격
- **학습 유형**: 지도 학습
- **문제 유형**: 회귀

타깃이 **수치형 값**이므로 분류가 아니라 회귀 문제다.

또한 여러 특성을 이용해 하나의 수치형 타깃을 예측하므로 **다중 회귀**(multiple regression)이자 **단변량 회귀**(univariate regression) 문제다.

### 데이터셋 기초 탐색

모델을 훈련하기 전에 데이터의 구조와 품질을 먼저 확인한다.

데이터셋 탐색에서는 다음을 살펴본다.

- 각 행과 열이 무엇을 의미하는가?
- 결측치는 있는가?
- 범주형 특성과 수치형 특성은 무엇인가?
- 특성의 값 범위와 분포는 어떠한가?
- 타깃 값에는 특별한 제한이나 이상한 점이 있는가?

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-05a.png?raw=true" width="350">
</div>

#### 범주형 특성

`ocean_proximity`는 해안과의 위치 관계를 나타내는 **범주형 특성**이다.
이 특성의 값은 `<1H OCEAN`, `INLAND`, `NEAR OCEAN`, `NEAR BAY`, `ISLAND`와 같은 **범주형 값**으로 구성된다.

| 특성값 | 설명 |
| :--- | :--- |
| <1H OCEAN | 해안에서 1시간 이내 |
| INLAND | 내륙 |
| NEAR OCEAN | 해안 근처 |
| NEAR BAY | 샌프란시스코의 Bay Area 구역 |
| ISLAND | 섬  |

#### 수치형 특성

나머지 특성은 수치형 특성이다.
수치형 특성에 대해서는 먼저 평균, 표준편차, 사분위수 등의 요약 통계를 살펴본다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/housing-describe.png?raw=true" width="100%">
</div>

히스토그램을 이용해 분포를 확인하는 것도 권장된다.

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/feature-histogram.png?raw=true" width="100%"></div></p>

히스토그램에서 몇 가지 중요한 사실을 확인할 수 있다.

- 특성마다 사용하는 단위와 스케일이 다르다.
- 일부 특성은 한쪽으로 치우친 분포를 보인다.
- 일부 특성은 값의 상한이 인위적으로 제한되어 있다.
- `total_bedrooms`에는 결측치가 존재한다.

이런 특징은 이후의 데이터 전처리 방법을 결정하는 근거가 된다.

### 훈련셋과 테스트셋

모델을 훈련하기 전에 전체 훈련 데이터를 **훈련셋**(training set)과 **테스트셋**(test set)으로 나눈다.

- **훈련셋**: 모델의 훈련과 모델 선택에 사용
- **테스트셋**: 최종적으로 선택된 모델의 일반화 성능을 평가하는 데 사용

> **테스트셋은 모델을 선택하거나 조정하는 과정에서 사용하지 않는다.**

#### 무작위 샘플링과 층화 샘플링

가장 간단한 방법은 전체 데이터에서 무작위로 샘플을 선택하는 **무작위 샘플링**(random sampling)이다.

그러나 데이터가 충분히 크지 않거나 중요한 집단의 비율을 유지해야 하는 경우에는 특정 집단이 훈련셋이나 테스트셋에 지나치게 많이 또는 적게 포함될 수 있다.

**층화 샘플링**(stratified sampling)은 중요한 특성을 기준으로 데이터를 여러 계층으로 나눈 뒤, 전체 데이터에서의 비율이 각 데이터셋에도 비슷하게 유지되도록 샘플을 선택하는 방법이다.

#### 중위소득 구간 활용

캘리포니아 주택 가격 데이터에서는 중위소득이 주택가격과 밀접하게 관련될 가능성이 있으므로, 
중위소득 구간을 5개로 구분한 다음에 중위소득 구간의 비율을 유지하도록 층화 샘플링을 사용할 수 있다.

중요한 것은 특정 샘플링 방법을 항상 사용하는 것이 아니라,

> **훈련셋과 테스트셋이 실제 데이터의 중요한 특성을 적절히 대표하는가?**

를 확인하는 것이다.

### 훈련셋 대상 EDA

훈련셋과 테스트셋을 나눈 뒤에는 **훈련셋만 이용해** 데이터의 관계를 탐색한다.
이유는 머신러닝 모델에 사용될 좋은 훈련셋으로 활용하는 방안을 모색하기 위해서인데,
테스트셋을 사용하면 미래에 발생할 데이터를 안다고 가정하는 결과를 초래하기 때문이다.

#### 상관관계

앞으로 훈련시킬 모델은 어떤 구역의 중위 주택가격을 제외한 다른 특성이 주어졌을 때
해당 구역의 중위 주택가격을 예측해야 한다.
따라서 중위 주택가격과 상관관계가 높은 특성을 미리 확인해볼 필요가 있다.

특성들 사이의 선형 상관관계를 피어슨 상관계수로 계산한다.
단, 수치형 특성만 대상으로 한다.
수치형 특성들 사이의 선형 관계는 **피어슨 상관계수**(Pearson correlation coefficient)로 확인할 수 있다.

상관계수는 -1에서 1 사이의 값을 가지며,

- 1에 가까울수록 강한 양의 선형 관계
- -1에 가까울수록 강한 음의 선형 관계
- 0에 가까울수록 약한 선형 관계

를 나타낸다.

상관계수가 0에 가깝다고 해서 두 특성 사이에 모든 종류의 관계가 없다는 뜻은 아님에 주의한다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-12c.png?raw=true" width="100%">
</div>

중위 주택가격과 중위소득의 상관계수가 0.68로 상당히 높다.
이는 중위소득이 높을수록 중위 주택가격도 높아지는 선형적 경향이 비교적 강하게 나타남을 의미한다.
아래 산점도가 이 사실을 잘 확인시켜준다. 

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-13.png?raw=true" width="500"></div>

## 데이터 준비에서 모델 평가까지

아래 내용을 체계적으로 살펴본다. 

- 데이터를 모델 학습에 적합한 형태로 전처리하고, 사이킷런이 제공하는 변환기를 이용하여 전처리 파이프라인을 구성한다.
- 이후 여러 모델을 훈련하고 성능을 비교한 다음, 검증 과정을 통해 적절한 모델을 선택하고 마지막으로 테스트셋을 이용해 최종 성능을 평가한다.

### 입력 데이터와 타깃

지도 학습 모델을 훈련하려면 훈련셋을 **입력 데이터**와 **타깃**으로 분리한다.

- 입력 데이터: 중위 주택가격 특성을 제외한 다른 특성으로 구성된 데이터
- 타깃: 중위 주택가격 특성으로만 구성된 데이터

이렇게 분리하면 이후의 전처리는 입력 데이터에 적용하고, 타깃은 모델이 예측해야 할 값으로 유지할 수 있다.

### 데이터 정제와 전처리

실제 데이터는 그대로 모델에 넣기 어려운 경우가 많다.

- 데이터 정제: 데이터의 오류, 결함 등 품질 문제를 찾아 수정하는 단계
- 데이터 전처리: 정제된 데이터를 머신러닝 모델이 학습할 수 있는 표현으로 변환하는 단계

캘리포니아 하우스 데이터에서는 특별한 데이터 오류 또는 결함이 없지만 다음 전처리가 필요하다.

- 결측치 처리
- 범주형 특성의 원-핫 인코딩
- 수치형 특성의 스케일링
- 필요에 따른 특성 변환과 특성 조합

전처리의 목적은 단순히 데이터를 '깨끗하게' 만드는 것이 아니라,

> **모델이 의미 있게 사용할 수 있는 형태로 데이터를 변환하는 것**

이다.

#### 결측치 처리

`total_bedrooms`에는 일부 값이 누락되어 있는데,
많은 머신러닝 모델은 결측치가 있는 훈련셋을 활용하지 못한다.

결측치를 처리하는 대표적인 방법은

- 해당 샘플 제거
- 해당 특성 제거
- 평균, 중앙값 등의 대표값으로 대체

하는 것이다.

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/null-value01.png?raw=true" width="100%"></div></p>

#### 원-핫 인코딩

머신러닝 모델은 일반적으로 문자열 형태의 범주형 값을 직접 사용하지 못한다.

`ocean_proximity`와 같은 범주형 특성은 **원-핫 인코딩**(one-hot encoding)을 이용해 각 범주형 값을 0과 1로 표현된 새로운 특성으로 변환할 수 있다.

이렇게 새롭게 생성된 특성을 **더미**(dummy) 특성이라 한다.

`ocean_proximity` 특성은 다섯 개의 범주로 구성되기에 원-핫 인코딩을 통해 다섯 개의 새로운 더미 특성을 생성하여
원래 특성 대신 모델 훈련에 활용한다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/one_hot01.png?raw=true" width="700">
</div>

#### 특성 스케일링

수치형 특성마다 값의 범위가 크게 다르면 일부 머신러닝 알고리즘의 훈련에 영향을 줄 수 있다.

특성 스케일링은 정규화 또는 표준화를 통해 수치형 특성을 일정 크기로 변환하는 과정을 의미한다.

| 용어 | 정의 |
|------|------|
| **정규화**(Normalization) | 데이터 값을 일정한 범위로 맞추는 일반적인 과정.<br>0과 1 사이로 맞추는 min-max 스케일링이 대표적. |
| **표준화**(Standardization) | 평균을 0, 표준편차를 1로 맞추는 과정. |


모든 모델이 스케일링에 똑같이 민감한 것은 아니지만, 여러 모델을 일관된 방식으로 비교할 때 유용하다.

#### 특성 변환

일부 수치형 특성은 한쪽으로 심하게 치우친 분포를 보인다.
이런 경우 로그 함수를 적용한 값으로 구성된 새로운 특성을 생성하는
로그 변환을 적용하여 분포의 치우침을 줄일 수 있다.

아래 그림은 구역별 인구로 구성된 `population` 특성값에 로그함수를 적용할 때 분포가 보다 균형잡히는 것을 잘 보여준다.

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-log_app.jpg?raw=true" width="500"></div></p>

### 변환기와 파이프라인

데이터 정제와 전처리의 모든 단계가 정확한 순서대로 진행되어야 한다.
사이킷런은 여러 변환기를 순차적으로 또는 병렬적으로 실행하는
파이프라인 기능을 지원한다.

> **파이프라인의 핵심은 훈련 데이터에 적용한 전처리를 새로운 데이터에도 동일한 방식으로 적용하는 것**이다.

### 모델 선택, 훈련, 평가

#### 다양한 모델 활용

전처리가 완료되면 여러 회귀 모델을 같은 조건에서 훈련시켜 서로 비교할 수 있다.

이 장에서는 다음 세 회귀 모델을 비교한다.

- `LinearRegression`
- `DecisionTreeRegressor`
- `RandomForestRegressor`

각 모델의 내부 원리는 뒤의 장에서 자세히 다룬다.
여기서는 **서로 다른 모델을 어떻게 비교하고 선택하는가**에 집중한다.

#### RMSE로 훈련 성능 확인

회귀 모델의 예측 오차는 **RMSE**(root mean squared error)라 불리는
평균 제곱근 오차로 평가할 수 있다.

RMSE는 예측 오차의 제곱의 평균값에 제곱근을 취한 값이며,
0에 가까울수록 실제값과 예측값의 차이가 작다는 뜻이다.

캘리포니아 주택가격으로 훈련된 세 예측 모델의 훈련셋에 대한 RMSE는 대략 다음과 같다.

| 모델 | 훈련셋 RMSE | 관찰 |
|---|---:|---|
| 선형 회귀 | 약 68,688 | 훈련 데이터에서도 오차가 큼 |
| 결정트리 | 0 | 훈련 데이터에는 완벽하게 맞음 |
| 랜덤 포레스트 | 약 17,474 | 선형 회귀보다 훈련 오차가 작음 |

하지만 **훈련셋 성능만으로는 어떤 모델이 새로운 데이터에서 잘 작동할지 판단할 수 없다.**
특히 결정트리의 RMSE가 0이라는 사실은 오히려 과대적합을 의심하게 한다.

#### 교차 검증으로 모델 비교하기

테스트셋은 최종 평가를 위해 남겨 두어야 한다.
그렇다면 훈련 과정에서 여러 모델을 어떻게 비교할 수 있을까?

**교차 검증**(cross-validation)은 훈련셋을 다시 여러 부분으로 나누어 모델을 반복해서 훈련하고 검증하는 방법이다.

아래 그림은 5-겹 교차 검증을 묘사한다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/cross-val10.png?raw=true" width="400">
</div>

교차 검증을 이용하면 한 번의 훈련 결과만 보는 것보다 모델의 일반화 성능을 더 안정적으로 비교할 수 있다.

아래 표는 5-겹 교차 검증을 통해 얻은 세 가지 모델의 성능(평균 RMSE)을 요약한다.

| 모델명 | 교차 검증 평균 RMSE | 성능 평가 요약 |
| :--- | :--- | :--- |
| **선형 회귀 모델** | 약 69,841 | 세 모델 중 가장 높은(나쁜) 오차를 보임 |
| **결정트리 회귀 모델** | 약 66,252 | 선형 회귀 모델보다 약간 낫지만 여전히 꽤 높은 오차를 보임 |
| **랜덤 포레스트 회귀 모델** | 약 47,224 | 훈련이 다소 오래 걸리지만 세 모델 중 성능이 가장 뛰어남 |

### 모델 하이퍼파라미터 미세 조정

모델의 훈련 방식을 조절하기 위해 사람이 미리 정하는 설정값을 **하이퍼파라미터**(hyperparameter)라고 한다. 반면 훈련을 통해 데이터에서 학습되는 값은 **모델 파라미터**(model parameter)다.

좋은 하이퍼파라미터 조합을 찾는 대표적인 방법은 다음과 같다.

* **그리드 탐색**(grid search): 지정한 조합을 체계적으로 확인
* **랜덤 탐색**(randomized search): 탐색 공간에서 일부 조합을 무작위로 확인

### 최적 모델 활용과 평가

가장 좋은 하이퍼파라미터를 선택해 모델 훈련을 마친 뒤 테스트셋을 사용한다.

최종 모델의 예측값과 실제 타깃을 비교하여 RMSE를 계산하고, 이를 **일반화 성능의 최종 추정치**로 사용한다.

테스트 결과를 보고 같은 테스트셋에 맞춰 모델을 다시 조정하면 최종 평가의 의미가 약해진다.

## 모델 훈련 원리 이해

앞에서는 여러 모델을 훈련하고 성능을 비교하는 머신러닝 프로젝트의 전체 과정을 살펴보았다.
이제 선형 회귀 모델을 이용하여 모델이 훈련 과정에서 실제로 무엇을 학습하는지 살펴본다.

선형 회귀는 구조가 단순하여 파라미터, 비용 함수, 경사하강법 등 모델 훈련의 핵심 개념을 설명하기에 적합하다.
특히 경사하강법을 이용한 파라미터 학습의 기본 원리는 로지스틱 회귀와 신경망 등 여러 머신러닝 모델에서도 활용된다.

**예제: 캘리포니아 주택 가격 예측**

캘리포니아 주택 가격을 예측하는 선형 회귀 모델은
구역별로 주어진 9개의 특성값을 13개로 변환한 다음에 그 지역의 중위 주택 가격을 예측한다.
즉, 1개의 편향과 함께 13개의 가중치를 활용한 아래 모양의 함수를 이용하여 예측값을 계산한다.

$$\hat y = \theta_0 + x_1 \cdot \theta_1 + \cdots + x_{13} \cdot \theta_{13}$$

* $\hat y$: 구역의 예측된 중위 주택 가격
* $x_i$: 구역의 $i$ 번째 특성값(위도, 경도, 중간소득, 가구당 인원 등)
* $\theta_0$: 편향
* $\theta_i$: $i$ 번째 특성에 대한 가중치.

:::{note} 기울기 vs. 가중치

입력 데이터에 포함된 샘플들의 특성이 하나일 때는 기울기 표현이 적절했지만 특성이 2개 이상일 때는 더 이상 적절하지 않으며,
대신 각 특성값에 가해지는 **가중치**<font size='2'>weight</font>라는 표현이 선호된다.
:::

### 선형 회귀 모델 예측값 계산

위 두 예제에서 설명한 선형 회귀 모델이 
예측값을 생성할 때 사용하는 선형 함수를 일반화하면 다음과 같다.

먼저 훈련셋에 포함된 샘플이 $n$ 개의 특성 $x_1$, $x_2$, ..., $x_n$을 갖는다고 가정할 때,
선형 회귀 모델은 아래 식을 이용하여 예측값을 계산한다.
$\theta_0$와 1을 곱해주는 이유는 다른 항과의 형식을 맞추기 위함이다.

$$\hat y = 1\cdot \theta_0 + x_1 \cdot \theta_1 + \cdots + x_n \cdot \theta_{n}$$

아래 이미지는 위 식을 시각화한다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/perceptron02.png" width="250"/>
</div>

**선형 회귀 모델의 파라미터: 편향과 가중치**

머신러닝 모델을 훈련하는 주요 목표는 입력값과 타깃 사이에 존재하는 숨은 관계를 찾아내는 것이다.
선형 회귀 모델은 입력 특성과 가중치 $\theta_i$의 선형 조합을 이용하여 예측값을 계산한다.
따라서 편향과 가중치는 훈련 데이터로부터 학습해야 하는 **모델 파라미터**다.

선형 회귀의 파라미터는 여러 방법으로 구할 수 있다.
여기서는 머신러닝 모델의 일반적인 훈련 원리를 이해하기 위해 **경사하강법**으로 파라미터를 학습하는 경우를 살펴본다.

### 머신러닝 모델 훈련의 목표

머신러닝 모델의 훈련은 타깃에 최대한 가까운 예측값을 계산하도록 모델 파라미터를 결정하는 과정이다.

$n$개의 특성을 사용하는 선형 회귀 모델은 편향 $\theta_0$과
$n$개의 가중치 $\theta_1, \ldots, \theta_n$, 즉 총 $n+1$개의 파라미터를 학습한다.

**비용 함수**

비용 함수<font size='2'>cost function</font>는 모델의 예측이 실제 타깃과 얼마나 차이 나는지를 수치로 나타낸다.
비용 함수의 값이 작을수록 예측 오차가 작다.

모델의 예측 오차를 측정하는 함수를 **손실 함수**(loss function) 또는 **비용 함수**(cost function)라고 한다.
이 함수가 계산한 값을 **손실값**(loss)이라고 한다.
회귀 모델에서는 대표적으로 MSE를 사용한다.

**MSE: 회귀 모델의 비용 함수**

회귀 모델의 경우 일반적으로 **평균 제곱 오차**<font size="2">mean squared error</font>(MSE)를
비용 함수로 사용한다.
아래 수식에서 $\hat y^{(i)}$와 $y^{(i)}$는 각각 $i$ 번째 샘플에 대한 예측값과 타깃을, 
$m$은 입력 데이터셋의 크기를 가리킨다.

$$\begin{align*}
\mathrm{MSE}(\mathbf{\theta})
&=
\frac{1}{m}
\sum_{i=1}^{m}
\left(
\hat y^{(i)} - y^{(i)}
\right)^2
\end{align*}
$$

**모델 훈련의 최종 목표**

회귀 모델의 훈련 목표는 훈련셋에서 $\mathrm{MSE}(\mathbf{\theta})$가 작아지도록
적절한 파라미터 $\mathbf{\theta}$를 찾는 것이다.

선형 회귀의 파라미터는 직접 계산하는 방법이나 경사하강법을 이용해 구할 수 있다.
예를 들어 사이킷런의 `LinearRegression`은 최소제곱 문제의 해를 계산하고,
`SGDRegressor`는 경사하강법을 이용해 파라미터를 반복적으로 업데이트한다.

여기서는 로지스틱 회귀와 신경망의 훈련에도 활용되는 **경사하강법의 기본 원리**에 집중한다.

### 경사하강법

경사하강법을 이해하려면 먼저 아래 개념들을 충분히 숙지해야 한다.

**배치**<font size='2'>batch</font>

모델이 한 번의 파라미터 업데이트에 사용하는 데이터 샘플의 묶음을 **배치**라고 한다.
한 배치에 포함되는 샘플의 수를 **배치 크기**<font size='2'>batch size</font>라고 한다.

**스텝**<font size='2'>step</font>

하나의 배치에 대해 예측값과 손실값을 계산하고,
손실값을 줄이는 방향으로 파라미터를 한 번 업데이트하는 과정을 **스텝**이라고 한다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/step.png" width="400"/>
</div>

**에포크**<font size='2'>epoch</font>

훈련셋의 모든 샘플을 한 번씩 사용하여 모델을 훈련하는 과정을 **에포크**라고 한다.

따라서 한 에포크는 여러 개의 스텝으로 구성될 수 있다.

모델 훈련은 여러 번의 에포크 동안 수행되며, **에포크가 반복될수록 파라미터가 점차 업데이트되면서 손실값이 줄어드는 방향으로 학습이 진행된다.**

**학습률($\eta$)**

훈련 스텝마다 파라미터 $\mathbf{\theta}$를 얼마나 크게 조정할지를 정하는 **하이퍼파라미터**다.

#### 선형 회귀 모델의 경사하강법

MSE를 비용 함수로 사용하는 선형 회귀 모델의 파라미터를 업데이트하는 스텝에서
사용되는 경사하강법은 아래 과정으로 구성된다.

1. 파라미터 벡터 $\mathbf{\theta}$를 0 또는 임의의 값으로 초기화한 후에 훈련을 시작한다.

1. 지정된 에포크만큼 또는 그레이디언트 벡터
    $\nabla_\mathbf{\theta} \textrm{MSE}(\mathbf{\theta})$가 충분히 작아질 때까지 
    아래 과정으로 구성된 훈련 스텝을 반복한다.

    * 하나의 배치에 대해 예측값 생성 후 손실값 $\mathrm{MSE}(\mathbf{\theta})$ 계산.
    * $\mathbf{\theta}$를 아래 점화식을 이용하여 업데이트:

    $$
    \theta^{(\text{new})} = \theta^{(\text{old})}\, -\, \eta\cdot \nabla_\theta \textrm{MSE}(\theta^{(\text{old})})
    $$

    위 식에서 $\eta$는 학습률, 
    $\theta^{(\text{old})}$는 이전 스텝을 통해 얻어진 파라미터 벡터, 
    $\theta^{(\text{new})}$는 업데이트된 파라미터 벡터를 가리킨다.

:::{note} 그레이디언트 벡터의 방향

그레이디언트 벡터 $\nabla_\mathbf{\theta} \mathrm{MSE}(\mathbf{\theta})$는
현재 위치에서 비용 함수 값이 가장 빠르게 증가하는 방향을 나타낸다.
따라서 적절한 학습률을 사용하여 그 반대 방향으로 파라미터를 이동시키면 MSE를 줄일 수 있다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/gradient01b.png" width="500"/>
</div>
:::

#### 학습률의 중요성

경사하강법에서는 그레이디언트가 알려주는 방향으로 파라미터를 이동시키되,
한 번에 얼마나 이동할지를 학습률이 결정한다.
선형 회귀 모델은 적절한 학습률로 경사하강법으로 적용할 경우
빠른 시간에 비용 함수가 전역 최소값을 갖도록 하는 $\hat{\theta}$ 에 수렴한다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-01.png" width="500"/>
</div>

**학습률이 너무 작은 경우**

비용 함수가 전역 최소값을 갖도록 하는 $\hat{\theta}$ 에 너무 느리게 수렴한다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-02.png" width="500"/>
</div>

**학습률이 너무 큰 경우**

비용 함수가 전역 최소값을 갖도록 하는 $\hat{\theta}$ 에 수렴하지 않고 발산한다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-03.png" width="500"/>
</div>

따라서 경사하강법을 사용하는 모델에서는 **적절한 학습률을 선택하는 것이 훈련의 속도와 안정성에 직접적인 영향을 준다.**
학습률은 모델이 데이터에서 학습하는 파라미터가 아니라 사람이 정하거나 탐색을 통해 선택하는 하이퍼파라미터다.

### 과소/과대 적합과 모델 규제

모델이 너무 단순하면 훈련 데이터의 중요한 패턴을 충분히 학습하지 못할 수 있다.
반대로 모델이 지나치게 복잡하면 훈련 데이터에만 지나치게 잘 맞고 새로운 데이터에서는 성능이 떨어질 수 있다.

일반적으로 어떤 복잡도의 모델이 가장 좋은지 미리 알 수 없다.
따라서 훈련 성능과 교차 검증 성능을 함께 비교한다.

* **과소 적합**(underfitting): 훈련 성능과 교차 검증 성능이 모두 좋지 않은 경우
* **과대 적합**(overfitting): 훈련 성능은 좋지만 교차 검증 성능이 상대적으로 많이 떨어지는 경우

**과소 적합 모델 개선법**

모델이 데이터의 패턴을 충분히 학습하지 못한다면 다음 방법을 고려할 수 있다.

- 보다 복잡한 모델 사용
- 모델이 활용할 수 있는 유용한 특성 추가 또는 특성 공학

**과대 적합 모델 개선법**

- 보다 단순한 모델 사용
- 모델 규제 적용
- 가능한 경우 훈련 데이터 추가